# Broadcast Join Speedup Test

This notebook tests the speedup of broadcast join with different numbers of raylets.

## 1. Setup: Import Libraries and Connect to Ray

In [1]:
import ray
import pandas as pd
import time

# Connect to Ray cluster
print("Connecting to Ray cluster...")
ray.init(address='auto')
print(f"Connected! Available CPUs: {ray.available_resources()['CPU']}")

2026-08-26 22:56:47,309	INFO worker.py:1833 -- Connecting to existing Ray cluster at address: 192.168.86.116:6379...
2026-08-26 22:56:47,323	INFO worker.py:2024 -- Connected to Ray cluster.


Connecting to Ray cluster...
Connected! Available CPUs: 4.0


## 2. Create Large States Table (50 US States)

In [2]:
# Create states table with all 50 US states
states = pd.DataFrame({
    'state_name': [
        'Alabama', 'Alaska', 'Arizona', 'Arkansas', 'California',
        'Colorado', 'Connecticut', 'Delaware', 'Florida', 'Georgia',
        'Hawaii', 'Idaho', 'Illinois', 'Indiana', 'Iowa',
        'Kansas', 'Kentucky', 'Louisiana', 'Maine', 'Maryland',
        'Massachusetts', 'Michigan', 'Minnesota', 'Mississippi', 'Missouri',
        'Montana', 'Nebraska', 'Nevada', 'New Hampshire', 'New Jersey',
        'New Mexico', 'New York', 'North Carolina', 'North Dakota', 'Ohio',
        'Oklahoma', 'Oregon', 'Pennsylvania', 'Rhode Island', 'South Carolina',
        'South Dakota', 'Tennessee', 'Texas', 'Utah', 'Vermont',
        'Virginia', 'Washington', 'West Virginia', 'Wisconsin', 'Wyoming'
    ],
    'state_id': list(range(1, 51))
})

print(f"States table shape: {states.shape}")
print(states.head())

States table shape: (50, 2)
   state_name  state_id
0     Alabama         1
1      Alaska         2
2     Arizona         3
3    Arkansas         4
4  California         5


## 3. Create Large Cities Table

Create a large cities table with many cities distributed across states.

In [18]:
# Create a large cities table with 100,000 cities
num_cities = 10000000

cities = pd.DataFrame({
    'city_id': list(range(1, num_cities + 1)),
    'state_id': [(i % 50) + 1 for i in range(num_cities)]
})

print(f"Cities table shape: {cities.shape}")
print(f"Total cities: {len(cities):,}")
print(cities.head(10))

Cities table shape: (10000000, 2)
Total cities: 10,000,000
   city_id  state_id
0        1         1
1        2         2
2        3         3
3        4         4
4        5         5
5        6         6
6        7         7
7        8         8
8        9         9
9       10        10


## 4. Define Broadcast Join Function

In [ ]:
@ray.remote
def join_partition(cities_partition, states_table):
    """Perform join on a partition with the broadcast states table."""
    result = cities_partition.merge(states_table, on='state_id', how='inner')
    return result


def broadcast_join(cities, states, num_raylets):
    """
    Perform broadcast join using specified number of raylets.
    
    Args:
        cities: Cities dataframe
        states: States dataframe (to be broadcast)
        num_raylets: Number of raylets to use for parallel processing
    
    Returns:
        Joined dataframe and elapsed time in seconds
    """
    start_time = time.time()
    
    # Broadcast states table
    states_ref = ray.put(states)
    
    # Partition cities table
    city_partitions = []
    partition_size = len(cities) // num_raylets
    
    for i in range(num_raylets):
        start_idx = i * partition_size
        if i == num_raylets - 1:
            end_idx = len(cities)
        else:
            end_idx = (i + 1) * partition_size
        
        partition = cities.iloc[start_idx:end_idx]
        city_partitions.append(partition)
    
    # Submit join tasks
    futures = [join_partition.remote(partition, states_ref) for partition in city_partitions]
    
    # Gather results
    results = ray.get(futures)
    
    # Concatenate results
    result = pd.concat(results, ignore_index=True)
    
    end_time = time.time()
    elapsed = end_time - start_time
    
    return result, elapsed

## 5. Test with 1 Raylet

In [ ]:
print("Testing with 1 raylet...")
result_1, time_1 = broadcast_join(cities, states, num_raylets=1)
print(f"Completed in {time_1:.4f} seconds")
print(f"Result shape: {result_1.shape}")
print(result_1.head())

Testing with 1 raylet...
Completed in 0.6577 seconds
Result shape: (10000000, 3)
   city_id  state_id  state_name
0        1         1     Alabama
1        2         2      Alaska
2        3         3     Arizona
3        4         4    Arkansas
4        5         5  California


## 6. Test with 2 Raylets

In [ ]:
print("Testing with 2 raylets...")
result_2, time_2 = broadcast_join(cities, states, num_raylets=2)
print(f"Completed in {time_2:.4f} seconds")
print(f"Speedup vs 1 raylet: {time_1/time_2:.2f}x")
print(f"Result shape: {result_2.shape}")

Testing with 2 raylets...
Completed in 0.7424 seconds
Speedup vs 1 raylet: 0.89x
Result shape: (10000000, 3)


## 7. Test with 3 Raylets

In [ ]:
print("Testing with 3 raylets...")
result_3, time_3 = broadcast_join(cities, states, num_raylets=3)
print(f"Completed in {time_3:.4f} seconds")
print(f"Speedup vs 1 raylet: {time_1/time_3:.2f}x")
print(f"Result shape: {result_3.shape}")

Testing with 3 raylets...
Completed in 10.8506 seconds
Speedup vs 1 raylet: 0.06x
Result shape: (10000000, 3)


## 8. Test with 4 Raylets

In [ ]:
print("Testing with 4 raylets...")
result_4, time_4 = broadcast_join(cities, states, num_raylets=4)
print(f"Completed in {time_4:.4f} seconds")
print(f"Speedup vs 1 raylet: {time_1/time_4:.2f}x")
print(f"Result shape: {result_4.shape}")

Testing with 4 raylets...
Completed in 0.7818 seconds
Speedup vs 1 raylet: 0.84x
Result shape: (10000000, 3)


## 9. Summary of Results

In [ ]:
import matplotlib.pyplot as plt

# Summary table
summary = pd.DataFrame({
    'Raylets': [1, 2, 3, 4],
    'Time (s)': [time_1, time_2, time_3, time_4],
    'Speedup': [1.0, time_1/time_2, time_1/time_3, time_1/time_4]
})

print("Performance Summary:")
print(summary.to_string(index=False))

# Plot speedup
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.plot(summary['Raylets'], summary['Time (s)'], marker='o', linewidth=2, markersize=8)
plt.xlabel('Number of Raylets')
plt.ylabel('Time (seconds)')
plt.title('Execution Time vs Number of Raylets')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(summary['Raylets'], summary['Speedup'], marker='o', linewidth=2, markersize=8, label='Actual')
plt.plot(summary['Raylets'], summary['Raylets'], linestyle='--', alpha=0.5, label='Ideal (Linear)')
plt.xlabel('Number of Raylets')
plt.ylabel('Speedup')
plt.title('Speedup vs Number of Raylets')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'matplotlib'

## 10. Cleanup

In [ ]:
# Uncomment to shutdown Ray
# ray.shutdown()